# Step 2a: Train COOLANT (Paper-faithful Simultaneous)

**Official COOLANT training: all 3 modules trained simultaneously per epoch**

This follows the official COOLANT repository approach:
- 3 separate optimizers (Similarity, CLIP, Detection)
- All modules updated every epoch
- Multi-task learning with shared representations

Prerequisites: Run `0_verify_labels.ipynb` then `1_preprocess.ipynb` first.

Input: `../processed_data/hdf5/vifactcheck_*.h5`
Output: `../../training/checkpoints/`

In [1]:
# Setup and imports
import sys
import os
import math
import random
import json
import warnings
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
import mlflow
import mlflow.pytorch
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# Project path (workflow/ -> notebooks/ -> project_root)
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [2]:
# Device and seeds
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"ðŸ”§ Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths
HDF5_DIR = project_root / "notebooks" / "processed_data" / "hdf5"
SAVE_DIR = project_root / "training" / "checkpoints"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"ðŸ“‚ HDF5 dir: {HDF5_DIR}")
print(f"ðŸ’¾ Save dir: {SAVE_DIR}")

ðŸ”§ Device: cuda
ðŸ“‚ HDF5 dir: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\processed_data\hdf5
ðŸ’¾ Save dir: g:\My Drive\Thesis_Final\fake-new-detection\training\checkpoints


In [3]:
# Load data using HDF5DatasetSimple with weighted sampling
from src.processing.hdf5_dataset import create_hdf5_dataloaders_from_files
from collections import Counter
import h5py

BATCH_SIZE = 32

train_path = HDF5_DIR / "vifactcheck_train.h5"
dev_path = HDF5_DIR / "vifactcheck_dev.h5"
test_path = HDF5_DIR / "vifactcheck_test.h5"

print("Loading data...")
loaders, datasets = create_hdf5_dataloaders_from_files(
    train_path=str(train_path),
    dev_path=str(dev_path),
    test_path=str(test_path),
    batch_size=BATCH_SIZE,
    num_workers=0,
    weighted_sampling=True,  # Balance classes in each batch
)

# Verify shapes
sample_text, sample_image, sample_label = datasets["train"][0]
print(f"\nSample shapes:")
print(f"  Text: {sample_text.shape} (expected: [768, 512])")
print(f"  Image: {sample_image.shape} (expected: [2048])")
print(f"  Label: {sample_label}")

# Verify label distribution in training data
with h5py.File(str(train_path), "r") as f:
    train_labels = f["labels"][:]
label_dist = Counter(train_labels.tolist())
print(f"\nTraining label distribution: {dict(label_dist)}")
assert len(label_dist) >= 2, f"ERROR: Only {len(label_dist)} class in training data!"

# Compute class weights for CrossEntropyLoss
n_train = len(train_labels)
class_weight = torch.tensor(
    [n_train / (2 * label_dist.get(i, 1)) for i in range(2)],
    dtype=torch.float32,
).to(DEVICE)
print(f"Class weights for loss: {class_weight}")
print(f"\nAll DataLoaders ready!")

Loading data...
HDF5DatasetSimple: 3376 samples from vifactcheck_train.h5
HDF5DatasetSimple: 465 samples from vifactcheck_dev.h5
HDF5DatasetSimple: 973 samples from vifactcheck_test.h5
  Weighted sampling enabled: class_weights={1: 1.084484420173466, 0: 12.836501901140684}

DataLoaders created from separate files:
  Train: 106 batches (3376 samples)
  Dev:   15 batches (465 samples)
  Test:  31 batches (973 samples)

Sample shapes:
  Text: torch.Size([768, 256]) (expected: [768, 512])
  Image: torch.Size([2048]) (expected: [2048])
  Label: 1

Training label distribution: {1: 3113, 0: 263}
Class weights for loss: tensor([6.4183, 0.5422], device='cuda:0')

All DataLoaders ready!


In [4]:
# Import and setup ResNetCOOLANT model
from src.models.resnet_coolant import (
    ResNetCOOLANT,
    patch_encoding,
    patch_clip_projection,
    patch_cnn_with_dropout,
)

# Dimensions
IMAGE_DIM = 2048
TEXT_EMBED = 768
TEXT_SEQ_LEN = 512

# Config
CONFIG = {
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,
    "h_dim": 64,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_epochs": 30,
    "patience": 5,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "seed": SEED,
    "device": str(DEVICE),
    "batch_size": BATCH_SIZE,
}

print("Config defined")

Config defined


In [5]:
# Create and patch model
model = ResNetCOOLANT(CONFIG)

# Patch encodings for 2048-dim image features
patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)

# Patch CLIP projections
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)

# Patch FastCNN with dropout
patch_cnn_with_dropout(
    model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn_with_dropout(
    model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn_with_dropout(
    model.detection_module.ambiguity_module.encoding.shared_text_encoding,
    TEXT_EMBED,
    CONFIG["dropout"],
)

# Move to device
model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"âœ… Model ready with {n_params:,} trainable parameters")

âœ… Model ready with 5,610,288 trainable parameters


In [ ]:
# make_pair_sim function (self-supervised: no labels needed)
def make_pair_sim(text, image, hard_negative_ratio=0.3):
    """
    Create matched/unmatched text-image pairs for similarity learning.

    Self-supervised: every article's own text-image is a matched pair,
    shuffled pairs are unmatched. No labels involved.
    """
    batch_size = text.size(0)

    n_anchors = max(2, batch_size // 2)
    anchor_indices = torch.randperm(batch_size)[:n_anchors]

    t_anchor = text[anchor_indices].clone()
    i_anchor = image[anchor_indices].clone()

    i_positive = i_anchor.clone()

    n_hard = int(n_anchors * hard_negative_ratio)
    n_random = n_anchors - n_hard

    i_negative = i_anchor.clone()
    if n_hard > 0:
        i_negative[:n_hard] = i_anchor[:n_hard].roll(1, dims=0)

    if n_random > 0:
        offset = random.randint(2, max(3, n_anchors // 2))
        i_negative[n_hard:] = i_anchor[n_hard:].roll(offset, dims=0)

    return t_anchor, i_positive, i_negative


print("make_pair_sim defined (self-supervised, no labels)")

In [8]:
# Create 3 separate optimizers (paper-faithful: one per module)
optim_sim = torch.optim.AdamW(
    model.similarity_module.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
optim_clip = torch.optim.AdamW(
    model.clip_module.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
optim_det = torch.optim.AdamW(
    model.detection_module.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

print(f"Optimizers created (lr={CONFIG['lr']}, wd={CONFIG['weight_decay']})")

Optimizers created (lr=0.001, wd=0.0001)


In [9]:
# Cosine annealing scheduler with warmup
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.current_step = 0
        self.base_lrs = [group["lr"] for group in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for i, base_lr in enumerate(self.base_lrs):
            if self.current_step < self.warmup_steps:
                lr = base_lr * self.current_step / self.warmup_steps
            else:
                progress = (self.current_step - self.warmup_steps) / (
                    self.total_steps - self.warmup_steps
                )
                lr = self.min_lr + (base_lr - self.min_lr) * 0.5 * (
                    1 + math.cos(math.pi * progress)
                )
            self.optimizer.param_groups[i]["lr"] = lr


steps_per_epoch = len(loaders["train"])
total_steps = steps_per_epoch * CONFIG["max_epochs"]
warmup_steps = int(0.05 * total_steps)

sch_sim = WarmupCosineScheduler(optim_sim, warmup_steps, total_steps)
sch_clip = WarmupCosineScheduler(optim_clip, warmup_steps, total_steps)
sch_det = WarmupCosineScheduler(optim_det, warmup_steps, total_steps)

print(f"âœ… Schedulers created")

âœ… Schedulers created


In [10]:
# Loss functions - with class weighting for imbalanced data
loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce = nn.CrossEntropyLoss(
    weight=class_weight,
    label_smoothing=CONFIG["label_smoothing"],
)
loss_ce_clip = nn.CrossEntropyLoss()  # Unweighted CE for CLIP contrastive loss
loss_kl = nn.KLDivLoss(reduction="batchmean")


def soft_xe(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, 1)).sum() / logits.size(0)


print(f"Loss functions created (CE class weights: {class_weight.cpu().tolist()})")

Loss functions created (CE class weights: [6.418251037597656, 0.5422422289848328])


In [ ]:
# Multi-task run_epoch function with per-class metrics
from sklearn.metrics import f1_score, precision_recall_fscore_support
import warnings as _warnings


def run_epoch(loader, train=True, epoch_num=None):
    if train:
        model.train()
    else:
        model.eval()

    tot_loss, tot_ok, tot_n = 0.0, 0, 0
    all_y, all_p = [], []
    epoch_sim_loss, epoch_clip_loss, epoch_det_loss = 0.0, 0.0, 0.0
    num_batches = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for text, image, label in tqdm(loader, desc="Train" if train else "Eval"):
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            label = label.to(DEVICE)
            bs = label.size(0)

            # Task 1a: Similarity (self-supervised, no labels)
            ft, fi_m, fi_u = make_pair_sim(text, image)
            ta_m, ia_m, _ = model.similarity_module(ft, fi_m)
            ta_u, ia_u, _ = model.similarity_module(ft, fi_u)
            t_cat = torch.cat([ta_m, ta_u])
            i_cat = torch.cat([ia_m, ia_u])
            y_cos = torch.cat(
                [
                    torch.ones(ta_m.size(0), device=DEVICE),
                    -torch.ones(ta_u.size(0), device=DEVICE),
                ]
            )
            ls = loss_cos(t_cat, i_cat, y_cos)
            epoch_sim_loss += ls.item()

            if train:
                optim_sim.zero_grad()
                ls.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.similarity_module.parameters(), CONFIG["grad_clip"]
                )
                optim_sim.step()
                sch_sim.step()

            # Task 1b: CLIP (self-supervised contrastive)
            text_pooled = text.mean(dim=2)
            ie, te = model.clip_module(image, text_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)

            ts, is_, _ = model.similarity_module(text, image)
            soft_m = is_ @ ts.T * math.exp(0.07)

            lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2
            ls2 = (
                soft_xe(logits, F.softmax(soft_m, 1))
                + soft_xe(logits.T, F.softmax(soft_m.T, 1))
            ) / 2
            l_clip = lc + 0.2 * ls2
            epoch_clip_loss += l_clip.item()

            if train:
                optim_clip.zero_grad()
                l_clip.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.clip_module.parameters(), CONFIG["grad_clip"]
                )
                optim_clip.step()
                sch_clip.step()

            # Task 2: Detection (supervised, uses labels)
            with torch.no_grad() if not train else torch.enable_grad():
                ie2, te2 = model.clip_module(image, text_pooled)
            det, attn, skl = model.detection_module(text, image, te2, ie2)
            ld = loss_ce(det, label) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )
            epoch_det_loss += ld.item()

            if train:
                optim_det.zero_grad()
                ld.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.detection_module.parameters(), CONFIG["grad_clip"]
                )
                optim_det.step()
                sch_det.step()

            pred = det.argmax(1)
            tot_ok += pred.eq(label).sum().item()
            tot_n += bs
            tot_loss += ld.item() * bs
            all_y.extend(label.cpu().numpy())
            all_p.extend(pred.cpu().numpy())
            num_batches += 1

    # Compute per-class metrics
    all_y_arr = np.array(all_y)
    all_p_arr = np.array(all_p)

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        macro_f1 = f1_score(all_y_arr, all_p_arr, average="macro", zero_division=0)
        prec, rec, f1, sup = precision_recall_fscore_support(
            all_y_arr, all_p_arr, average=None, zero_division=0
        )

    metrics = {
        "loss": tot_loss / max(tot_n, 1),
        "acc": tot_ok / max(tot_n, 1),
        "macro_f1": macro_f1,
        "sim_loss": epoch_sim_loss / max(num_batches, 1),
        "clip_loss": epoch_clip_loss / max(num_batches, 1),
        "det_loss": epoch_det_loss / max(num_batches, 1),
    }

    # Per-class metrics (real=0, fake=1)
    for i, name in enumerate(["real", "fake"]):
        if i < len(prec):
            metrics[f"{name}_precision"] = prec[i]
            metrics[f"{name}_recall"] = rec[i]
            metrics[f"{name}_f1"] = f1[i]

    return metrics, all_y, all_p


print("run_epoch defined (similarity/CLIP self-supervised, detection supervised)")

In [12]:
# MLflow setup - use local path without spaces to avoid URL-encoding issue
MLFLOW_DIR = Path(os.path.expanduser("~")) / "mlflow_runs"
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow_db = str(MLFLOW_DIR / "mlflow.db").replace("\\", "/")
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db}")

EXPERIMENT_NAME = "vietnamese-coolant-training"
mlflow.set_experiment(EXPERIMENT_NAME)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"coolant-{timestamp}"
mlflow.start_run(run_name=run_name)

mlflow.log_params(CONFIG)
print(f"MLflow tracking: {MLFLOW_DIR}")
print(f"MLflow run started: {run_name}")

MLflow tracking: C:\Users\tungh\mlflow_runs
MLflow run started: coolant-20260416_220529


In [13]:
# Training loop with early stopping on macro F1
best_val_f1 = 0.0
best_val_loss = float("inf")
best_val_acc = 0.0
patience_counter = 0
history = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
}

print(f"Starting training for max {CONFIG['max_epochs']} epochs...")
print(f"   Early stopping patience: {CONFIG['patience']} (on macro F1)")
print(f"   Class weights: {class_weight.cpu().tolist()}\n")

for epoch in range(1, CONFIG["max_epochs"] + 1):
    train_metrics, _, _ = run_epoch(loaders["train"], train=True, epoch_num=epoch)
    val_metrics, val_y, val_p = run_epoch(loaders["dev"], train=False, epoch_num=epoch)

    history["train_loss"].append(train_metrics["loss"])
    history["train_acc"].append(train_metrics["acc"])
    history["train_f1"].append(train_metrics["macro_f1"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])
    history["val_f1"].append(val_metrics["macro_f1"])

    mlflow.log_metrics(
        {
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_real_recall": val_metrics.get("real_recall", 0),
            "val_fake_recall": val_metrics.get("fake_recall", 0),
            "lr_sim": optim_sim.param_groups[0]["lr"],
        },
        step=epoch,
    )

    print(f"Epoch [{epoch:02d}/{CONFIG['max_epochs']}]")
    print(f"  Train: loss={train_metrics['loss']:.4f} acc={train_metrics['acc']:.4f} F1={train_metrics['macro_f1']:.4f}")
    print(f"  Val:   loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} F1={val_metrics['macro_f1']:.4f}")
    print(f"  Val per-class: real_rec={val_metrics.get('real_recall', 0):.3f} fake_rec={val_metrics.get('fake_recall', 0):.3f}")

    # Print confusion matrix
    cm = confusion_matrix(val_y, val_p)
    print(f"  Confusion matrix: {cm.tolist()}")

    # Early stopping on macro F1 (more meaningful for imbalanced data)
    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_val_loss = val_metrics["loss"]
        best_val_acc = val_metrics["acc"]
        patience_counter = 0

        checkpoint_path = SAVE_DIR / "best_model.pth"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optim_sim_state_dict": optim_sim.state_dict(),
                "optim_clip_state_dict": optim_clip.state_dict(),
                "optim_det_state_dict": optim_det.state_dict(),
                "val_loss": best_val_loss,
                "val_acc": best_val_acc,
                "val_f1": best_val_f1,
                "config": CONFIG,
            },
            checkpoint_path,
        )
        mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
        print(f"  >> New best macro_f1={best_val_f1:.4f}, saved checkpoint")
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/{CONFIG['patience']})")

    if patience_counter >= CONFIG["patience"]:
        print(f"\nEarly stopping triggered at epoch {epoch}")
        break
    print()

print(f"\nTraining complete!")
print(f"   Best val macro_f1: {best_val_f1:.4f}")
print(f"   Best val_loss: {best_val_loss:.4f}")
print(f"   Best val_acc: {best_val_acc:.4f}")

Starting training for max 30 epochs...
   Early stopping patience: 5 (on macro F1)
   Class weights: [6.418251037597656, 0.5422422289848328]



Eval: 100%|██████████| 15/15 [01:25<00:00,  5.70s/it]


Epoch [01/30]
  Train: loss=0.5371 acc=0.5323 F1=0.5168
  Val:   loss=0.4391 acc=0.4774 F1=0.3231
  Val per-class: real_rec=1.000 fake_rec=0.000
  Confusion matrix: [[222, 0], [243, 0]]
  >> New best macro_f1=0.3231, saved checkpoint



Eval: 100%|██████████| 15/15 [01:24<00:00,  5.65s/it]


Epoch [02/30]
  Train: loss=0.1797 acc=0.6967 F1=0.6685
  Val:   loss=0.8480 acc=0.4774 F1=0.3231
  Val per-class: real_rec=1.000 fake_rec=0.000
  Confusion matrix: [[222, 0], [243, 0]]
  -> No improvement (1/5)



Train:   8%|▊         | 9/106 [00:57<10:20,  6.40s/it]


KeyboardInterrupt: 

In [ ]:
# Load best model
checkpoint = torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"âœ… Loaded best model from epoch {checkpoint['epoch']}")

In [ ]:
# Test evaluation with per-class metrics
print("\nEvaluating on test set...")
test_metrics, test_y, test_p = run_epoch(loaders["test"], train=False)

print(f"\nTest Results:")
print(f"   Loss: {test_metrics['loss']:.4f}")
print(f"   Accuracy: {test_metrics['acc']:.4f}")
print(f"   Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"   Real recall: {test_metrics.get('real_recall', 0):.4f}")
print(f"   Fake recall: {test_metrics.get('fake_recall', 0):.4f}")

report = classification_report(test_y, test_p, target_names=["Real", "Fake"], digits=4)
print("\nClassification Report:")
print(report)

cm = confusion_matrix(test_y, test_p)
print("Confusion Matrix:")
print(cm)

mlflow.log_metrics({
    "test_loss": test_metrics["loss"],
    "test_acc": test_metrics["acc"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_real_recall": test_metrics.get("real_recall", 0),
    "test_fake_recall": test_metrics.get("fake_recall", 0),
})

report_path = SAVE_DIR / "test_report.txt"
with open(report_path, "w") as f:
    f.write("Test Classification Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(report)
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(cm))
    f.write(f"\n\nMacro F1: {test_metrics['macro_f1']:.4f}")

mlflow.log_artifact(str(report_path), artifact_path="results")
print("\nTest report saved")

In [ ]:
# End MLflow run
mlflow.end_run()
print("âœ… MLflow run ended")